# 04 — Quality control

Run `src.qc.qc_file` over the dataset. Two passes:
1. **Cheap pass** (~30 s) — metadata only.
2. **Deep pass** (~25 min) — loads `ce_foot` / `ce_torso` and checks for NaN/Inf and flat-line spectrograms.

Produces `outputs/metrics/qc_report.csv` and `excluded_files.csv`.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)

from tqdm.notebook import tqdm
from src.data_loader import iter_trials
from src.qc import qc_file


## 1. Smoke test on one file

In [2]:
trials = list(iter_trials())
rec = qc_file(trials[0].path, deep=True)
for k, v in rec.items():
    print(f"  {k:18s}  {v}")


  path                /Users/ziad.boussedra/Desktop/Master Thesis/thesis-parkinson-radar/data/fisc_005/test1/trial1/stft_data.mat
  ok                  True
  fails               []
  doppler_min         -794.872802734375
  doppler_max         799.8720092773438
  doppler_n           320
  duration_s          7.543081283569336
  t_n                 12068
  ce_foot_shape       320x12068
  ce_torso_shape      320x12068
  n_fails             0
  fails_str           


## 2. Cheap pass over all trials

In [3]:
from tqdm.auto import tqdm
cheap = []
for tp in tqdm(trials, desc="cheap QC"):
    r = qc_file(tp.path, deep=False)
    r.update({"subject_id": tp.subject_id, "group": tp.group,
              "test": tp.test, "trial": tp.trial})
    cheap.append(r)
df_cheap = pd.DataFrame(cheap)
print(f"ok={df_cheap['ok'].sum()}  failed={(~df_cheap['ok']).sum()}")
df_cheap.loc[~df_cheap["ok"], ["subject_id", "test", "trial", "fails_str"]].head(20)


cheap QC:   0%|          | 0/348 [00:00<?, ?it/s]

ok=348  failed=0


,subject_id,test,trial,fails_str


## 3. Distribution of failure reasons

In [4]:
if (~df_cheap["ok"]).any():
    reasons = []
    for fails_str in df_cheap.loc[~df_cheap["ok"], "fails_str"]:
        reasons.extend(f.split(":", 1)[0] for f in fails_str.split("; ") if f)
    ser = pd.Series(reasons).value_counts()
    print(ser)
    fig, ax = plt.subplots(figsize=(8, 3))
    ser.plot.barh(ax=ax, color="#e66")
    ax.set_xlabel("count")
    ax.set_title("QC failure reasons (cheap pass)")
    plt.tight_layout(); plt.show()
else:
    print("No cheap-check failures.")


No cheap-check failures.


## 4. Deep pass (optional, slow)

Set `RUN_DEEP = True` to load every CE array and run NaN/Inf + dynamic-range checks. ~25 minutes on a Mac with an SSD.

In [5]:
from tqdm.auto import tqdm
RUN_DEEP = False

if RUN_DEEP:
    deep = []
    for tp in tqdm(trials, desc="deep QC"):
        r = qc_file(tp.path, deep=True)
        r.update({"subject_id": tp.subject_id, "group": tp.group,
                  "test": tp.test, "trial": tp.trial})
        deep.append(r)
    df = pd.DataFrame(deep)
else:
    df = df_cheap
print(f"final: ok={df['ok'].sum()}  failed={(~df['ok']).sum()}")


final: ok=348  failed=0


## 5. Visualise a flagged trial (if any)

In [6]:
if (~df["ok"]).any():
    from src.data_loader import load_trial
    bad_row = df.loc[~df["ok"]].iloc[0]
    print("Inspecting:", bad_row["fails_str"])
    bad_path = next(t for t in trials if t.subject_id == bad_row["subject_id"]
                    and t.test == bad_row["test"] and t.trial == bad_row["trial"]).path
    try:
        tr = load_trial(bad_path)
        fig, axes = plt.subplots(1, 2, figsize=(13, 3))
        for ax, key in zip(axes, ["ce_foot", "ce_torso"]):
            ax.imshow(np.log1p(tr[key]), aspect="auto", origin="lower", cmap="magma")
            ax.set_title(f"{bad_row['subject_id']}/{bad_row['test']}/{bad_row['trial']} — {key}")
        plt.tight_layout(); plt.show()
    except Exception as e:
        print(f"could not load: {e}")
else:
    print("No flagged trials to inspect.")


No flagged trials to inspect.


## 6. Save report and exclusion list

In [7]:
(ROOT / "outputs" / "metrics").mkdir(parents=True, exist_ok=True)
df.to_csv(ROOT / "outputs" / "metrics" / "qc_report.csv", index=False)
df.loc[~df["ok"]].to_csv(ROOT / "excluded_files.csv", index=False)
print(f"qc_report.csv:    {len(df)} rows")
print(f"excluded_files.csv: {(~df['ok']).sum()} rows")


qc_report.csv:    348 rows
excluded_files.csv: 0 rows


### Next step

If any trials are excluded, pass the path list to `preprocessing.preprocess_dataset(excluded_paths=...)` in Notebook 06 so they don't make it into the .npy cache.